# Causal Feature Attribution: What Actually Drives Netflix Churn?

## The Business Problem

Netflix's product team asks: **"What is driving subscriber churn?"**

The data science team has a churn prediction model with strong AUC. They run SHAP and report that *support tickets* is the #1 driver. The product team proposes investing \$10M in a "ticket deflection" initiative.

**But is this the right action?**

Prediction tells you WHAT will happen. Causal attribution tells you WHY it happens and WHAT TO DO about it.

In this notebook we show that naively acting on SHAP values can lead to misguided interventions — and how causal attribution via Double Machine Learning (DML) identifies the truly actionable drivers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score
import shap
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

## Step 1: Simulate Users from a Causal Graph

We generate 15,000 users according to a **known causal structure**. Having the true data-generating process lets us verify whether each method recovers the correct drivers.

### The Causal Graph

```
                 user_enthusiasm (unmeasured)
                /           |            \
               v            v             v
        watch_hours   genre_diversity   social_usage
              |             |              |
              v             v              v
         ─────────────> CHURN <────────────
              ^                            ^
              |                            |
         account_age ──────────────────────┘
          (confounder)

        dissatisfaction ──────> support_tickets
              |                   (COLLIDER)
              v
            CHURN
```

### What each arrow means

| Arrow | Meaning |
|-------|---------|
| `user_enthusiasm → watch_hours/genre_diversity/social_usage` | Enthusiastic users naturally watch more, explore more genres, and use social features more. This is an **unmeasured confounder** linking the three behaviors. |
| `watch_hours → churn (β = −0.05)` | **Causal**: Each additional weekly watch hour reduces churn probability. Intervening to increase watch hours *will* reduce churn. |
| `genre_diversity → churn (β = −0.03)` | **Causal**: Exploring more genres reduces churn. |
| `social_usage → churn (β = −0.02)` | **Causal**: Social feature usage reduces churn, but with the smallest effect. |
| `account_age → watch_hours, churn` | **Confounder**: Older accounts watch more *and* churn less independently. Naive analysis inflates watch_hours' apparent effect. |
| `dissatisfaction → support_tickets` | Dissatisfied users file more tickets. |
| `dissatisfaction → churn` | Dissatisfied users churn more. |
| `support_tickets` | **COLLIDER** — caused by dissatisfaction, does NOT cause churn. Tickets and churn share a common cause but tickets have zero causal effect on churn. |

In [ ]:
n = 15_000

# --- Unmeasured common cause ---
user_enthusiasm = np.random.normal(0, 1, n)

# --- Confounder: account age (months) ---
account_age = np.random.exponential(24, n)

# --- Behavioral features (driven by enthusiasm + noise) ---
watch_hours = 5 + 2 * user_enthusiasm + 0.05 * account_age + np.random.normal(0, 2, n)
watch_hours = np.clip(watch_hours, 0, None)

genre_diversity = 3 + 1.5 * user_enthusiasm + np.random.normal(0, 1.5, n)
genre_diversity = np.clip(genre_diversity, 1, 15).astype(int)

social_usage = 2 + 1.0 * user_enthusiasm + np.random.normal(0, 1.5, n)
social_usage = np.clip(social_usage, 0, None)

# --- Dissatisfaction (independent latent factor) ---
dissatisfaction = np.random.normal(0, 1, n)

# --- Support tickets: caused BY dissatisfaction, NOT a cause of churn ---
support_tickets = np.clip(np.round(1.5 * dissatisfaction + np.random.normal(0, 0.5, n)), 0, 10).astype(int)

# --- CHURN: the true causal model ---
churn_logit = (
    0.5
    - 0.05 * watch_hours       # causal: more watching → less churn
    - 0.03 * genre_diversity   # causal: more diversity → less churn
    - 0.02 * social_usage      # causal: more social → less churn
    - 0.01 * account_age       # confounder effect on churn
    + 0.30 * dissatisfaction   # dissatisfaction → churn (but NOT via tickets)
)
churn_prob = 1 / (1 + np.exp(-churn_logit))
churn = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

df = pd.DataFrame({
    'watch_hours': np.round(watch_hours, 1),
    'genre_diversity': genre_diversity,
    'social_usage': np.round(social_usage, 1),
    'support_tickets': support_tickets,
    'account_age': np.round(account_age, 1),
    'churn': churn,
})

print(f"Dataset: {len(df):,} users")
print(f"Churn rate: {df['churn'].mean():.1%}")
print(f"\nFeature summary:")
df.describe().round(2)

## Step 2: The DAG — Making Assumptions Explicit

Before running any model, we draw the causal graph. This is not optional — it encodes our domain assumptions about *why* variables are related.

The critical structural features:

1. **`watch_hours`, `genre_diversity`, `social_usage`** are genuine causes of churn — intervening on them changes churn.
2. **`account_age`** is a confounder — it affects both watch_hours and churn. If we don't control for it, we overestimate watch_hours' causal effect.
3. **`support_tickets`** is a **collider** on the path `dissatisfaction → tickets ← (nothing else)` and a non-causal correlate of churn. Tickets and churn are both caused by dissatisfaction, but tickets don't cause churn. Conditioning on tickets (e.g., including them in a model) can *create* spurious associations.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 7.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Causal DAG: Netflix Churn Drivers', fontsize=16, fontweight='bold', pad=20)

nodes = {
    'User\nEnthusiasm\n(unmeasured)': (5, 7),
    'Watch\nHours': (2, 4.5),
    'Genre\nDiversity': (5, 4.5),
    'Social\nUsage': (8, 4.5),
    'CHURN': (5, 1.5),
    'Account\nAge': (0.5, 2.5),
    'Dissatis-\nfaction': (9.5, 2.5),
    'Support\nTickets': (9.5, 5.5),
}

colors = {
    'User\nEnthusiasm\n(unmeasured)': '#D3D3D3',
    'Watch\nHours': '#4CAF50',
    'Genre\nDiversity': '#4CAF50',
    'Social\nUsage': '#4CAF50',
    'CHURN': '#F44336',
    'Account\nAge': '#FF9800',
    'Dissatis-\nfaction': '#D3D3D3',
    'Support\nTickets': '#9C27B0',
}

for name, (x, y) in nodes.items():
    box = mpatches.FancyBboxPatch(
        (x - 0.8, y - 0.5), 1.6, 1.0,
        boxstyle='round,pad=0.1', facecolor=colors[name],
        edgecolor='black', linewidth=1.5, alpha=0.85
    )
    ax.add_patch(box)
    ax.text(x, y, name, ha='center', va='center', fontsize=9, fontweight='bold')

causal_edges = [
    ('User\nEnthusiasm\n(unmeasured)', 'Watch\nHours'),
    ('User\nEnthusiasm\n(unmeasured)', 'Genre\nDiversity'),
    ('User\nEnthusiasm\n(unmeasured)', 'Social\nUsage'),
    ('Watch\nHours', 'CHURN'),
    ('Genre\nDiversity', 'CHURN'),
    ('Social\nUsage', 'CHURN'),
]
confounder_edges = [
    ('Account\nAge', 'Watch\nHours'),
    ('Account\nAge', 'CHURN'),
]
collider_edges = [
    ('Dissatis-\nfaction', 'Support\nTickets'),
    ('Dissatis-\nfaction', 'CHURN'),
]

arrow_kw = dict(arrowstyle='->', lw=2, mutation_scale=20)

for src, dst in causal_edges:
    sx, sy = nodes[src]
    dx, dy = nodes[dst]
    ax.annotate('', xy=(dx, dy + 0.5), xytext=(sx, sy - 0.5),
                arrowprops=dict(**arrow_kw, color='#2E7D32'))

for src, dst in confounder_edges:
    sx, sy = nodes[src]
    dx, dy = nodes[dst]
    ax.annotate('', xy=(dx - 0.7, dy + 0.3 if dy > sy else dy - 0.3),
                xytext=(sx + 0.8, sy + 0.3 if sy < dy else sy - 0.3),
                arrowprops=dict(**arrow_kw, color='#E65100', linestyle='dashed'))

for src, dst in collider_edges:
    sx, sy = nodes[src]
    dx, dy = nodes[dst]
    ax.annotate('', xy=(dx - 0.7 if dx < sx else dx + 0.7, dy + 0.3 if dy > sy else dy - 0.3),
                xytext=(sx - 0.0, sy + 0.5 if sy < dy else sy - 0.5),
                arrowprops=dict(**arrow_kw, color='#6A1B9A', linestyle='dotted'))

legend_elements = [
    mpatches.Patch(facecolor='#4CAF50', label='Causal driver (actionable)'),
    mpatches.Patch(facecolor='#FF9800', label='Confounder'),
    mpatches.Patch(facecolor='#9C27B0', label='Collider (NOT a cause)'),
    mpatches.Patch(facecolor='#D3D3D3', label='Unmeasured / latent'),
    mpatches.Patch(facecolor='#F44336', label='Outcome'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

## Step 3: The Standard ML Approach — XGBoost + SHAP

This is what most data science teams do: train a predictive model, then use SHAP to explain it.

Let's see what SHAP tells us about the "drivers" of churn.

In [ ]:
features = ['watch_hours', 'genre_diversity', 'social_usage', 'support_tickets', 'account_age']
X = df[features]
y = df['churn']

model = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42
)
model.fit(X, y)

y_pred_proba = model.predict_proba(X)[:, 1]
auc = roc_auc_score(y, y_pred_proba)
print(f"XGBoost AUC: {auc:.3f}")
print(f"\nThe model predicts churn well. Now let's see what SHAP says is 'driving' it...")

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0), index=features
).sort_values(ascending=False)

print("SHAP Feature Importance (mean |SHAP value|):")
print("=" * 45)
for rank, (feat, val) in enumerate(mean_abs_shap.items(), 1):
    marker = " ← MISLEADING!" if feat == 'support_tickets' else ""
    print(f"  #{rank}: {feat:20s}  {val:.4f}{marker}")

fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#E53935' if f == 'support_tickets' else '#1976D2' for f in mean_abs_shap.index]
ax.barh(mean_abs_shap.index[::-1], mean_abs_shap.values[::-1], color=bar_colors[::-1])
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('SHAP Feature Importance — What the Prediction Model Says', fontweight='bold')
ax.axvline(x=0, color='grey', linewidth=0.5)

for i, (feat, val) in enumerate(zip(mean_abs_shap.index[::-1], mean_abs_shap.values[::-1])):
    if feat == 'support_tickets':
        ax.text(val + 0.002, i, '⚠ NOT a causal driver!', va='center', fontsize=11,
                color='#E53935', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 4: Why SHAP Is Misleading Here

SHAP ranks **support_tickets** as a top predictor. This is *statistically correct* — tickets genuinely help predict churn, because both are caused by dissatisfaction:

```
dissatisfaction ──→ support_tickets
       |                              
       └──────────→ CHURN
```

**Support tickets are a COLLIDER / non-causal proxy.** They are a downstream *symptom* of dissatisfaction — the same latent factor that causes churn. Tickets don't cause churn; they *co-occur* with it.

### Why acting on SHAP here is dangerous

If the product team invests \$10M in "ticket deflection" (making it harder to file tickets, or auto-resolving them):
- Tickets go down ✓
- Churn stays the same ✗ (dissatisfaction unchanged)
- \$10M wasted

### The core issue

| Concept | SHAP | Causal Attribution |
|---------|------|--------------------|
| What it measures | How much a feature helps the *model predict* | How much *changing* a feature changes the *outcome* |
| Confounders | Exploits all correlations | Controls for confounders |
| Colliders | Can rank them highly | Correctly assigns zero effect |
| Actionability | No guarantee | Direct — estimates are interventional |

## Step 5: Why Causal Attribution, Not SHAP

SHAP answers: *"Which features does the model rely on for prediction?"*

Causal attribution answers: *"Which features, if we intervened on them, would change the outcome?"*

These are different questions. They give different answers when:
1. **Confounders** inflate the apparent importance of a feature (account_age → watch_hours)
2. **Colliders/proxies** appear predictive but have zero causal effect (support_tickets)
3. **Mediators** are counted twice by SHAP but need careful handling causally

For **driver analysis** — where the goal is to decide *what to change* — we need causal attribution.

## Step 6: Causal Approach — Double Machine Learning (DML)

**Double Machine Learning** (Chernozhukov et al., 2018) estimates the causal effect of a treatment $T$ on outcome $Y$ while flexibly controlling for confounders $W$:

1. **Residualize Y**: Fit a flexible model $\hat{Y} = f(W)$, compute $\tilde{Y} = Y - \hat{Y}$
2. **Residualize T**: Fit a flexible model $\hat{T} = g(W)$, compute $\tilde{T} = T - \hat{T}$
3. **Regress residuals**: $\tilde{Y} = \theta \cdot \tilde{T} + \epsilon$

The key insight: after removing what confounders predict about both $Y$ and $T$, the remaining variation in $T$ is "as-if random" — and its association with the remaining variation in $Y$ is the causal effect $\theta$.

We use cross-fitting (out-of-fold predictions) to avoid overfitting bias.

In [ ]:
def estimate_dml(df, treatment_col, outcome_col, confounder_cols, n_splits=5):
    """Estimate the causal effect of treatment on outcome using DML with cross-fitting."""
    Y = df[outcome_col].values
    T = df[treatment_col].values
    W = df[confounder_cols].values

    from sklearn.model_selection import KFold

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    Y_residuals = np.zeros(len(Y))
    T_residuals = np.zeros(len(T))

    for train_idx, test_idx in kf.split(W):
        model_y = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
        model_y.fit(W[train_idx], Y[train_idx])
        Y_residuals[test_idx] = Y[test_idx] - model_y.predict(W[test_idx])

        model_t = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
        model_t.fit(W[train_idx], T[train_idx])
        T_residuals[test_idx] = T[test_idx] - model_t.predict(W[test_idx])

    theta = np.sum(T_residuals * Y_residuals) / np.sum(T_residuals ** 2)

    residuals = Y_residuals - theta * T_residuals
    var_theta = np.sum(T_residuals ** 2 * residuals ** 2) / (np.sum(T_residuals ** 2) ** 2)
    se_theta = np.sqrt(var_theta)

    return theta, se_theta

print("DML function defined. Now estimating causal effects for each candidate driver...")

In [ ]:
candidate_drivers = ['watch_hours', 'genre_diversity', 'social_usage', 'support_tickets']

confounders_map = {
    'watch_hours': ['account_age', 'genre_diversity', 'social_usage'],
    'genre_diversity': ['account_age', 'watch_hours', 'social_usage'],
    'social_usage': ['account_age', 'watch_hours', 'genre_diversity'],
    'support_tickets': ['account_age', 'watch_hours', 'genre_diversity', 'social_usage'],
}

dml_results = []

print("Causal Effect Estimates via Double Machine Learning")
print("=" * 65)
print(f"{'Feature':<22} {'Causal Effect':>14} {'Std Error':>12} {'95% CI':>22}")
print("-" * 65)

for driver in candidate_drivers:
    confounders = confounders_map[driver]
    theta, se = estimate_dml(df, driver, 'churn', confounders)

    ci_low = theta - 1.96 * se
    ci_high = theta + 1.96 * se

    dml_results.append({
        'feature': driver,
        'causal_effect': theta,
        'se': se,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'significant': (ci_low > 0) or (ci_high < 0),
    })

    sig_marker = "***" if abs(theta / se) > 2.58 else "**" if abs(theta / se) > 1.96 else ""
    print(f"  {driver:<20} {theta:>+14.5f} {se:>12.5f}   [{ci_low:>+.5f}, {ci_high:>+.5f}] {sig_marker}")

dml_df = pd.DataFrame(dml_results)
print("\n*** p < 0.01, ** p < 0.05")
print("\nTrue causal effects: watch_hours=-0.05, genre_diversity=-0.03, social_usage=-0.02, support_tickets=0.00")

## Step 7: SHAP vs. Causal Rankings — The Comparison

Now we compare what SHAP says are the top drivers versus what causal analysis says.

In [ ]:
shap_ranks = mean_abs_shap.rank(ascending=False).astype(int)

causal_ranks = dml_df.set_index('feature')['causal_effect'].abs().rank(ascending=False).astype(int)

comparison = pd.DataFrame({
    'Feature': candidate_drivers,
    'SHAP Importance': [f"{mean_abs_shap[f]:.4f}" for f in candidate_drivers],
    'SHAP Rank': [shap_ranks[f] for f in candidate_drivers],
    'Causal Effect (DML)': [f"{dml_df[dml_df['feature']==f]['causal_effect'].values[0]:+.5f}" for f in candidate_drivers],
    'Causal Rank': [causal_ranks[f] for f in candidate_drivers],
    'Divergence': [
        'HIGH — SHAP inflated!' if f == 'support_tickets' else
        'Moderate — confounder bias' if f == 'account_age' else
        'Low'
        for f in candidate_drivers
    ],
})

print("\n" + "=" * 95)
print("SHAP vs CAUSAL RANKING COMPARISON")
print("=" * 95)
print(comparison.to_string(index=False))
print("\n⚠ KEY DIVERGENCE: support_tickets ranks high in SHAP but has ZERO causal effect.")
print("  Acting on SHAP's recommendation to reduce tickets would waste resources.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shap_sorted = mean_abs_shap[candidate_drivers].sort_values()
colors_shap = ['#E53935' if f == 'support_tickets' else '#1976D2' for f in shap_sorted.index]
axes[0].barh(shap_sorted.index, shap_sorted.values, color=colors_shap)
axes[0].set_title('SHAP Rankings\n(Predictive Importance)', fontweight='bold')
axes[0].set_xlabel('Mean |SHAP Value|')

causal_sorted = dml_df.set_index('feature')['causal_effect'].abs()[candidate_drivers].sort_values()
colors_causal = ['#E53935' if f == 'support_tickets' else '#4CAF50' for f in causal_sorted.index]
axes[1].barh(causal_sorted.index, causal_sorted.values, color=colors_causal)
axes[1].set_title('Causal Rankings\n(DML Estimated Effect)', fontweight='bold')
axes[1].set_xlabel('|Causal Effect|')

for ax in axes:
    ax.axvline(x=0, color='grey', linewidth=0.5)

plt.suptitle('Prediction ≠ Causation: Rankings Can Diverge Dramatically',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 8: Actionability Matrix

Causal effect alone isn't enough — we also need to consider **feasibility of intervention**. A feature with a large causal effect that's impossible to change is less useful than one with a moderate effect that's easy to nudge.

We combine causal effect size with a domain-informed feasibility score to create a 2×2 prioritization matrix.

In [ ]:
actionability = pd.DataFrame({
    'feature': ['watch_hours', 'genre_diversity', 'social_usage', 'support_tickets'],
    'causal_impact': [abs(dml_df[dml_df['feature']==f]['causal_effect'].values[0]) for f in candidate_drivers],
    'feasibility': [0.7, 0.85, 0.5, 0.9],
    'intervention': [
        'Personalized recommendations,\nauto-play, "continue watching"',
        'Genre exploration prompts,\n"Because you watched..."',
        'Social features, watch parties,\nshared watchlists',
        'Ticket deflection\n(NO causal impact!)'
    ],
})

fig, ax = plt.subplots(figsize=(12, 8))

for _, row in actionability.iterrows():
    color = '#E53935' if row['feature'] == 'support_tickets' else '#1976D2'
    marker = 'X' if row['feature'] == 'support_tickets' else 'o'
    size = 300 if row['feature'] == 'support_tickets' else 200

    ax.scatter(row['feasibility'], row['causal_impact'], s=size, c=color,
               marker=marker, zorder=5, edgecolors='black', linewidth=1.5)
    ax.annotate(row['intervention'], (row['feasibility'], row['causal_impact']),
                textcoords='offset points', xytext=(15, 10), fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.7),
                arrowprops=dict(arrowstyle='->', color='grey'))

median_impact = actionability[actionability['feature'] != 'support_tickets']['causal_impact'].median()
ax.axhline(y=median_impact, color='grey', linestyle='--', alpha=0.5)
ax.axvline(x=0.65, color='grey', linestyle='--', alpha=0.5)

ax.text(0.85, max(actionability['causal_impact']) * 0.95, 'HIGH PRIORITY\n(High Impact, Easy)',
        ha='center', fontsize=11, fontweight='bold', color='#2E7D32',
        bbox=dict(facecolor='#E8F5E9', alpha=0.8, boxstyle='round'))
ax.text(0.55, max(actionability['causal_impact']) * 0.95, 'WORTH INVESTIGATING\n(High Impact, Hard)',
        ha='center', fontsize=10, color='#F57F17',
        bbox=dict(facecolor='#FFF8E1', alpha=0.8, boxstyle='round'))
ax.text(0.85, min(actionability['causal_impact']) * 1.1 + 0.002, 'QUICK WINS\n(Low Impact, Easy)',
        ha='center', fontsize=10, color='#1565C0',
        bbox=dict(facecolor='#E3F2FD', alpha=0.8, boxstyle='round'))
ax.text(0.55, min(actionability['causal_impact']) * 1.1 + 0.002, 'DEPRIORITIZE\n(Low Impact, Hard)',
        ha='center', fontsize=10, color='#757575',
        bbox=dict(facecolor='#F5F5F5', alpha=0.8, boxstyle='round'))

ax.set_xlabel('Feasibility of Intervention', fontsize=13)
ax.set_ylabel('Causal Impact (|DML Effect|)', fontsize=13)
ax.set_title('Actionability Matrix: Causal Impact × Feasibility',
             fontsize=14, fontweight='bold')
ax.set_xlim(0.4, 1.0)

legend_elements = [
    plt.scatter([], [], c='#1976D2', marker='o', s=100, label='Causal driver'),
    plt.scatter([], [], c='#E53935', marker='X', s=100, label='Non-causal (SHAP artifact)'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=11)

plt.tight_layout()
plt.show()

## Step 9: Validation — Simulating Interventions

The ultimate test: what happens if we actually *intervene* on different features?

We simulate two scenarios:
1. **SHAP-recommended intervention**: Reduce support tickets by 50% (the top SHAP feature)
2. **Causal-recommended intervention**: Increase watch hours by 2 hours/week (the top causal driver)

Because we know the true data-generating process, we can compute the *true* effect of each intervention.

In [ ]:
baseline_churn_rate = churn_prob.mean()

# --- Intervention 1: SHAP-recommended — reduce support tickets by 50% ---
# Tickets have ZERO causal effect, so this changes nothing in the true model
intervention_tickets_logit = churn_logit.copy()  # tickets not in the formula → no change
intervention_tickets_prob = 1 / (1 + np.exp(-intervention_tickets_logit))
ticket_intervention_rate = intervention_tickets_prob.mean()

# --- Intervention 2: Causal-recommended — increase watch hours by 2 ---
intervention_watch_logit = churn_logit - 0.05 * 2  # 2 extra hours × true causal effect
intervention_watch_prob = 1 / (1 + np.exp(-intervention_watch_logit))
watch_intervention_rate = intervention_watch_prob.mean()

# --- Intervention 3: Also try genre diversity (+2 genres) ---
intervention_genre_logit = churn_logit - 0.03 * 2
intervention_genre_prob = 1 / (1 + np.exp(-intervention_genre_logit))
genre_intervention_rate = intervention_genre_prob.mean()

print("INTERVENTION SIMULATION (True Causal Effects)")
print("=" * 60)
print(f"  Baseline churn rate:                {baseline_churn_rate:.4f} ({baseline_churn_rate:.1%})")
print()
print(f"  ❌ SHAP Intervention:")
print(f"     Reduce support tickets by 50%")
print(f"     New churn rate: {ticket_intervention_rate:.4f} ({ticket_intervention_rate:.1%})")
print(f"     Churn reduction: {(baseline_churn_rate - ticket_intervention_rate):.4f} ({(baseline_churn_rate - ticket_intervention_rate)/baseline_churn_rate:.1%} relative)")
print()
print(f"  ✅ Causal Intervention A:")
print(f"     Increase watch hours by +2 hrs/week")
print(f"     New churn rate: {watch_intervention_rate:.4f} ({watch_intervention_rate:.1%})")
print(f"     Churn reduction: {(baseline_churn_rate - watch_intervention_rate):.4f} ({(baseline_churn_rate - watch_intervention_rate)/baseline_churn_rate:.1%} relative)")
print()
print(f"  ✅ Causal Intervention B:")
print(f"     Increase genre diversity by +2 genres")
print(f"     New churn rate: {genre_intervention_rate:.4f} ({genre_intervention_rate:.1%})")
print(f"     Churn reduction: {(baseline_churn_rate - genre_intervention_rate):.4f} ({(baseline_churn_rate - genre_intervention_rate)/baseline_churn_rate:.1%} relative)")

fig, ax = plt.subplots(figsize=(10, 5))

scenarios = ['Baseline\n(No intervention)', 'SHAP Recommendation:\nReduce tickets 50%',
             'Causal Rec. A:\n+2 watch hrs/wk', 'Causal Rec. B:\n+2 genres explored']
rates = [baseline_churn_rate, ticket_intervention_rate, watch_intervention_rate, genre_intervention_rate]
colors = ['#757575', '#E53935', '#4CAF50', '#2196F3']

bars = ax.bar(scenarios, rates, color=colors, edgecolor='black', linewidth=1.2, width=0.6)

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{rate:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Churn Rate', fontsize=13)
ax.set_title('Simulated Intervention Outcomes\n(SHAP vs Causal Recommendations)',
             fontsize=14, fontweight='bold')
ax.set_ylim(0, max(rates) * 1.25)

ax.annotate('Zero effect!\nTickets are not\na cause of churn.',
            xy=(1, ticket_intervention_rate), xytext=(1.3, max(rates) * 1.1),
            fontsize=10, color='#E53935', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#E53935'),
            bbox=dict(boxstyle='round', facecolor='#FFEBEE'))

plt.tight_layout()
plt.show()

## Key Takeaways

### 1. Prediction ≠ Causation
A feature that is highly predictive of an outcome is not necessarily a cause of it. Support tickets are an excellent *predictor* of churn (because they share a common cause: dissatisfaction) but have zero *causal* effect.

### 2. Acting on SHAP Can Backfire
SHAP explains the *model*, not the *world*. If the model exploits a spurious correlation (collider bias, confounding), SHAP will faithfully report that feature as important. Product teams that act on these rankings risk:
- Wasting resources on interventions that don't move the needle
- Missing the actually actionable levers
- Eroding trust in data science when the intervention fails

### 3. DAGs Make Assumptions Explicit
The causal graph is not optional. It encodes:
- Which variables are confounders (must be controlled)
- Which are colliders (must NOT be conditioned on)
- Which are mediators (require careful handling)

Without a DAG, you cannot determine the right adjustment set, and any "causal" estimate is just a dressed-up correlation.

### 4. The DML Workflow for Driver Analysis
1. **Draw the DAG** with domain experts
2. **Identify confounders** for each candidate driver
3. **Estimate causal effects** via DML (residualize, regress)
4. **Rank by causal impact**, not predictive importance
5. **Combine with feasibility** in an actionability matrix
6. **Validate** with A/B tests where possible

### Methods Used
- **Double Machine Learning (DML)** — Chernozhukov et al., 2018
- **DAG-based identification** — Pearl, 2009
- **SHAP** (for contrast) — Lundberg & Lee, 2017

### Software
- In production, use **EconML** (`LinearDML`, `CausalForestDML`) and **DoWhy** for the full identification → estimation → refutation pipeline.